# B6.2 — Family-disjoint multi-task U-Net

Use a GPU runtime. This notebook builds or reuses the exact B6.1 archive, trains seeds 42/43/44, selects checkpoints and thresholds on validation families only, compares B2 on the same B6 tiles, and stores resume-safe outputs in Google Drive. The historical test families are development confirmation, not the untouched B9 holdout.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive')
PERSISTENT_ROOT = DRIVE_ROOT / 'ADVLSI2_B6_2'
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
PERSISTENT_ROOT

In [ ]:
import subprocess, sys

REPO = Path('/content/ADVLSI2_Project_updated')
BRANCH = 'agent/b6-2-multitask-unet'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/nocleo/ADVLSI2_Project_updated.git', str(REPO)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO, check=True)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())

In [ ]:
import hashlib, json, shutil

DATASET_CACHE = PERSISTENT_ROOT / 'b6_localization_dataset.zip'
if not DATASET_CACHE.exists():
    subprocess.run([sys.executable, 'scripts/build_b6_localization_dataset.py'], cwd=REPO, check=True)
    shutil.copy2(REPO / 'training_datasets/b6_localization_dataset.zip', DATASET_CACHE)

digest = hashlib.sha256(DATASET_CACHE.read_bytes()).hexdigest()
print(f'Dataset: {DATASET_CACHE} ({DATASET_CACHE.stat().st_size / 1e6:.1f} MB)')
print(f'SHA-256: {digest}')

In [ ]:
B2_CHECKPOINTS = DRIVE_ROOT / 'ADVLSI2_B2/b2_baselines/checkpoints'
required = [B2_CHECKPOINTS / f'unseen_layout_v1_seed_{seed}.pth' for seed in (42, 43, 44)]
missing = [str(path) for path in required if not path.exists()]
assert not missing, 'Missing authoritative B2 checkpoints: ' + ', '.join(missing)

OUTPUT_DIR = PERSISTENT_ROOT / 'b6_multitask_unet'
command = [
    sys.executable, 'scripts/run_b6_multitask_unet.py',
    '--dataset', str(DATASET_CACHE),
    '--output-dir', str(OUTPUT_DIR),
    '--b2-checkpoint-dir', str(B2_CHECKPOINTS),
    '--device', 'cuda',
]
print(' '.join(command))
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
from IPython.display import Markdown, display
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
assert summary['status'] == 'complete'
assert summary['official_result'] is True
assert summary['seeds'] == [42, 43, 44]
assert summary['untouched_final_holdout_used'] is False
display(Markdown((OUTPUT_DIR / 'README.md').read_text()))
summary['acceptance']

In [ ]:
from google.colab import files
archive = shutil.make_archive('/content/ADVLSI2_B6_2_results', 'zip', OUTPUT_DIR)
files.download(archive)